In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def build_pswm(contexts_df, background_freqs=None, log_base="log2"):
    aa_alphabet = sorted("ACDEFGHIKLMNPQRSTVWY")
    n_positions = 15  
    
    if background_freqs is None:
        background_freqs = {
            'A': 0.0825, 'R': 0.0553, 'N': 0.0406, 'D': 0.0545, 'C': 0.0137,
            'Q': 0.0393, 'E': 0.0675, 'G': 0.0707, 'H': 0.0227, 'I': 0.0596,
            'L': 0.0966, 'K': 0.0584, 'M': 0.0242, 'F': 0.0386, 'P': 0.0470,
            'S': 0.0656, 'T': 0.0534, 'W': 0.0108, 'Y': 0.0292, 'V': 0.0687
        }
        
    counts = np.ones((len(aa_alphabet), n_positions))
    aa_to_index = {aa: idx for idx, aa in enumerate(aa_alphabet)}
    
    for context_str in contexts_df["context"]:
        for pos_idx, aa in enumerate(context_str):
            if aa in aa_to_index:
                counts[aa_to_index[aa], pos_idx] += 1
                
    pspm = counts / counts.sum(axis=0)
    bg_vector = np.array([background_freqs[aa] for aa in aa_alphabet]).reshape(-1, 1)
    
    if log_base == "log2":
        pswm_array = np.log2(pspm / bg_vector)
    elif log_base == "ln":
        pswm_array = np.log(pspm / bg_vector)
    else:
        raise ValueError("log_base must be either 'log2' or 'ln'")
    
    position_labels = [f"P{i}" for i in range(-13, 3) if i != 0]
    pswm_df = pd.DataFrame(pswm_array, index=aa_alphabet, columns=position_labels)
    
    return pswm_df

if __name__ == "__main__":
    input_dir = Path("../data/prepared")
    try:
        train, validation, test = get_cv_split(data, test_fold=1)
        contexts, excluded = extract_cleavage_contexts(train)
        pswm_matrix = build_pswm(contexts)
        print("Success")
        display(pswm_matrix.head())
    except NameError:
        print("Functions not defined in memory yet")
